MeshAPI Feature Tour
This notebook walks through every feature MeshAPI offers, one at a time -- not just the ones our other notebooks happened to use. Each section is: a plain-English explanation, the smallest working code example, and the real output from running it against a live key. See research.md for the full written inventory this notebook is based on.

# MeshAPI Feature Tour

This notebook explores the major features available through MeshAPI,
one feature at a time.

Each section contains:

1. A plain-English explanation
2. A minimal working example
3. The actual output from MeshAPI

The goal is to understand MeshAPI as a unified AI gateway rather
than using only the features required by the RAG + multi-agent project.

## Cost and Scope

Most text examples use inexpensive models.

Image, video, and audio generation may incur small usage costs,
so those examples are kept minimal.

Some dashboard-level features require a dashboard session token
rather than the `rsk_...` API key.

BYOK is not demonstrated because it requires external provider
credentials configured through the MeshAPI dashboard.

MCP Server and CLI are discussed but are not executed as Python API calls.

In [1]:
import os
from getpass import getpass

from dotenv import load_dotenv
load_dotenv()

import json

from meshapi import MeshAPI, MeshAPIError

MESHAPI_BASE_URL = os.getenv("MESHAPI_BASE_URL", "https://api.meshapi.ai")
MESHAPI_TOKEN = os.getenv("MESH_API_KEY") or os.getenv("MESHAPI_TOKEN") or getpass("MeshAPI token (rsk_...): ")

client = MeshAPI(base_url=MESHAPI_BASE_URL, token=MESHAPI_TOKEN)
RAW_HEADERS = {"Authorization": f"Bearer {MESHAPI_TOKEN}", "Content-Type": "application/json"}


def show(obj, indent=2):
    """Pretty-print any SDK response -- they're all Pydantic models."""
    if hasattr(obj, "model_dump_json"):
        print(obj.model_dump_json(indent=indent))
    else:
        print(json.dumps(obj, indent=indent, default=str))


print("Client ready.")


Client ready.


Pick a couple of models to use throughout

In [2]:
FAST_MODEL = "openai/gpt-4o-mini"
SMART_MODEL = "mistral/mistral-large-3-675b-instruct"

print("FAST_MODEL  =", FAST_MODEL)
print("SMART_MODEL =", SMART_MODEL)

FAST_MODEL  = openai/gpt-4o-mini
SMART_MODEL = mistral/mistral-large-3-675b-instruct


1. Talking to models
The core of the gateway: send messages, get replies. Everything in this section works on /v1/chat/completions or its close cousins.

1.1 Chat Completions
The basic building block -- send a list of messages, get one reply back.

In [3]:
from meshapi import ChatCompletionParams, ChatMessage

resp = client.chat.completions.create(

    ChatCompletionParams(

        model=FAST_MODEL,

        messages=[ChatMessage(role="user", content="In one sentence, what is an AI gateway?")],

        max_tokens=60,

    )

)

print(resp.choices[0].message.content)


An AI gateway is a system or platform that facilitates the integration, management, and deployment of artificial intelligence models and services, enabling seamless communication between AI applications and various data sources or devices.


1.2 Streaming
Get the reply token-by-token as it's generated, instead of waiting for the whole thing.

In [5]:
for chunk in client.chat.completions.stream(

    ChatCompletionParams(model=FAST_MODEL, messages=[ChatMessage(role="user", content="Count from 1 to 50.")])

):

    if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:

        print(chunk.choices[0].delta.content, end="", flush=True)


Sure! Here is the count from 1 to 50:

1, 2, 3, 4, 5, 6, 7, 8, 9, 10,  
11, 12, 13, 14, 15, 16, 17, 18, 19, 20,  
21, 22, 23, 24, 25, 26, 27, 28, 29, 30,  
31, 32, 33, 34, 35, 36, 37, 38, 39, 40,  
41, 42, 43, 44, 45, 46, 47, 48, 49, 50.  


1.3 Tool / Function calling
Let the model call one of your Python functions when it needs real data it doesn't already know.

In [6]:
from meshapi import Tool, ToolFunction

WEATHER_TOOL = Tool(

    type="function",

    function=ToolFunction(

        name="get_weather",

        description="Get the current weather for a city.",

        parameters={"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},

    ),

)



resp = client.chat.completions.create(

    ChatCompletionParams(

        model=FAST_MODEL,

        messages=[ChatMessage(role="user", content="What's the weather in Paris?")],

        tools=[WEATHER_TOOL],

        tool_choice="auto",

        max_tokens=100,

    )

)

call = resp.choices[0].message.tool_calls[0]

print("Model wants to call:", call.function.name, "with args:", call.function.arguments)


Model wants to call: get_weather with args: {"city":"Paris"}


1.4 Structured outputs
Force the reply into a JSON shape you define with a Pydantic model, instead of free-form text. The installed SDK version doesn't have a convenience .parse() method -- so we build the JSON schema ourselves and validate the reply with Pydantic, which is what .parse() would do internally anyway (this is also exactly what LangChain's create_agent does under the hood in notebook 2).

In [7]:
from pydantic import BaseModel

class Country(BaseModel):

    country: str

    capital: str

    population_millions: float



resp = client.chat.completions.create(

    ChatCompletionParams(

        model=FAST_MODEL,

        messages=[ChatMessage(role="user", content="Give me facts about France.")],

        response_format={"type": "json_schema", "json_schema": {"name": "Country", "schema": Country.model_json_schema()}},

    )

)

country = Country.model_validate_json(resp.choices[0].message.content)

print(type(country).__name__, "->", country)


Country -> country='France' capital='Paris' population_millions=67.0


1.5 Compare
Ask 2+ models the same question in one call, side by side.

In [8]:
from meshapi import CompareParams

result = client.compare.create(

    CompareParams(

        models=[FAST_MODEL, SMART_MODEL],

        messages=[ChatMessage(role="user", content="In 5 words or less, what is a gateway?")],

    )

)

show(result)


{
  "comparison_id": "cmp_dcdc20dc213d0417ae37",
  "object": "compare.completion",
  "created": 1788727905,
  "models": [
    "openai/gpt-4o-mini",
    "mistral/mistral-large-3-675b-instruct"
  ],
  "results": [
    {
      "model": "openai/gpt-4o-mini",
      "response_body": {
        "id": "chatcmpl-ELE7yaiZSP7bFzsGl8j06pq0dyF4o",
        "object": "chat.completion",
        "created": 1788727902,
        "model": "gpt-4o-mini-2024-07-18",
        "choices": [
          {
            "index": 0,
            "message": {
              "role": "assistant",
              "content": "Access point to a network.",
              "refusal": null,
              "annotations": []
            },
            "logprobs": null,
            "finish_reason": "stop"
          }
        ],
        "usage": {
          "prompt_tokens": 19,
          "completion_tokens": 6,
          "total_tokens": 25,
          "prompt_tokens_details": {
            "cached_tokens": 0,
            "audio_tokens": 0
 

Feature Tool Yet :-

MeshAPI Feature Tour
│
├── 1.1 Chat Completions
├── 1.2 Streaming
├── 1.3 Tool / Function Calling
├── 1.4 Structured Outputs
└── 1.5 Compare

1.6 Model discovery :- Model discovery allows applications to retrieve the models exposed by MeshAPI programmatically, making model selection more dynamic instead of relying entirely on hard-coded model names.

In [10]:
all_models = client.models.list()


print("total models:", len(all_models))



total models: 988


1.7 Error handling

In [11]:
try:

    client.chat.completions.create(

        ChatCompletionParams(model="not-a-real-provider/not-a-real-model", messages=[ChatMessage(role="user", content="hi")])

    )

except MeshAPIError as e:

    print(f"Caught it cleanly -> status={e.status} code={e.error_code}")

    print("message:", str(e))

Caught it cleanly -> status=404 code=model_not_found
message: Model 'not-a-real-provider/not-a-real-model' is not supported or is invalid. Did you mean 'cohere/command-a'?


1.8 Responses API:- The Responses API is an API interface for generating model responses, providing a more unified way to build modern AI interactions beyond the traditional Chat Completions interface.

In [12]:
from meshapi import ResponsesParams

r = client.responses.create(ResponsesParams(model=FAST_MODEL, input="Say pong in one word."))

print("status:", r.status)

print("text:", r.output[0]["content"][0]["text"])


status: completed
text: Pong!


1.9 Auto Router :- Instead of manually choosing which AI model should answer a request, Auto Router automatically chooses an appropriate model for you.

In [13]:
from meshapi import RouterSelectParams

choice = client.router.select(RouterSelectParams(messages=[ChatMessage(role="user", content="Write a haiku about the ocean")]))

print("router would pick:", choice.model)



resp = client.chat.completions.create(

    ChatCompletionParams(model="auto", messages=[ChatMessage(role="user", content="What is 12*8? Just the number.")], max_tokens=20)

)

print("actually picked:", resp.model, "-> reply:", resp.choices[0].message.content)

if resp.usage.classifier_tokens:

    print("classifier overhead (tokens):", resp.usage.classifier_tokens, "-- you're billed for this too")

else:

    print("classifier overhead: not reported for this provider/response (field is optional, sometimes None)")


router would pick: meta-llama/llama-4-scout
actually picked: anthropic/claude-haiku-4.5 -> reply: 96
classifier overhead: not reported for this provider/response (field is optional, sometimes None)


!. Talking to the models is done 